# LMSYS — Qwen2.5-7B + QLoRA — **TRAINING** notebook (1 of 2)

This is the ~10-hour run that produces the model. It does **not** predict on the test set —
see the "Why two notebooks" note below.

| | |
|---|---|
| Backbone | `Qwen/Qwen2.5-7B-Instruct` — ungated (Apache-2.0), so no HF token needed |
| Quantization | 4-bit NF4, double-quant, **fp16** compute (T4 has no bf16) |
| Adapter | LoRA `r=16, alpha=32` on all 7 projections + trainable `score` head |
| Input | prompt + response A + response B in **one** sequence, all conversation turns |
| Pooling | last non-pad token (standard for decoder classifiers) |
| Metric | multiclass log loss |
| Output | `/kaggle/working/qwen_lora_adapter/` — attach this to the inference notebook |

### Why two notebooks

On one T4, measured against the real model size:

```
training   ~0.42 samples/s  ->  one full epoch of 57,477 rows = 38 hours
inference  ~1.88 samples/s  ->  25,000 hidden test rows       = 3.7 hours
```

Kaggle caps a **submission** run at 9h, so train + infer cannot share one notebook. Splitting also
means a training crash never costs you the inference, and you can re-run inference cheaply.

### What this is NOT

It will not reach the 0.83 at the top of the leaderboard. In one ~10h session this sees roughly
**15,000 of 57,477 rows** — about a quarter of one epoch. Expect **~0.95–1.00**. Teams at 0.90 and
below used several sessions, larger models, or better hardware. The estimate is honest, not a target.

### Design for a long run

Six consecutive Kaggle failures went into this: an environment self-test runs in the first 30 seconds,
a hard deadline stops training cleanly, and the adapter is checkpointed every validation so a crash
at hour 9 still leaves you a usable model.

---
# 1 · Environment setup and self-test

Runs before anything expensive. If the Kaggle image has drifted, this fails in seconds.

In [ ]:
import importlib.util
import importlib.metadata
import subprocess
import sys


def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)


def _version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except Exception:
        return None


# peft>=0.15 raises if torchao is present but older than 0.16. We never use torchao,
# and peft treats it as absent-is-fine, so upgrade it or remove it.
if importlib.util.find_spec("torchao") is not None:
    _tv = _version("torchao")
    print(f"torchao present: {_tv}")
    try:
        from packaging.version import parse as _parse
        _too_old = _tv is not None and _parse(_tv) < _parse("0.16.0")
    except Exception:
        _too_old = True
    if _too_old:
        print("torchao too old for peft -> upgrading ...")
        _pip("-U", "torchao>=0.16.0")
        if _version("torchao") == _tv:
            print("upgrade did not take -> uninstalling torchao (unused here)")
            subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"],
                           check=False)

for _pkg in ("peft", "bitsandbytes", "accelerate"):
    if importlib.util.find_spec(_pkg) is None:
        print(f"Installing {_pkg} ...")
        _pip(_pkg)

import gc
import json
import math
import os
import random
import time
import warnings
from dataclasses import dataclass

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Sampler

from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss, accuracy_score

import transformers
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)

import bitsandbytes
import peft
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

NOTEBOOK_T0 = time.time()

print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("torchao      :", _version("torchao"))

---
# 2 · Configuration

In [ ]:
@dataclass
class CFG:
    seed: int = 42

    # ---- model ----
    # Ungated Apache-2.0 checkpoint: no HuggingFace token, no license click-through.
    # Swap for "google/gemma-2-9b-it" or "meta-llama/Meta-Llama-3-8B" only if you have
    # accepted their licences and set HF_TOKEN as a Kaggle secret.
    model_name: str = "Qwen/Qwen2.5-7B-Instruct"
    num_labels: int = 3
    max_length: int = 768

    # ---- character budgets used before tokenisation (~3.4 chars/token) ----
    prompt_chars: int = 600
    response_chars: int = 1000

    # ---- LoRA ----
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # ---- optimisation ----
    batch_size: int = 2               # per step; drop to 1 if you OOM
    gradient_accumulation_steps: int = 8   # effective batch 16
    eval_batch_size: int = 8
    learning_rate: float = 1e-4
    weight_decay: float = 0.0
    warmup_ratio: float = 0.03
    max_grad_norm: float = 0.3        # QLoRA convention
    label_smoothing: float = 0.0
    epochs: int = 1

    # ---- validation ----
    validation_size: float = 0.05     # 5% of 57k ~ 2,900 rows
    max_val_samples: int = 1500       # full-length eval is expensive; cap it
    eval_every_steps: int = 250       # optimiser steps between validations

    # ---- runtime budget ----
    # Kaggle allows 12h per session. Stop training well before that so the adapter
    # is saved and the notebook exits cleanly.
    time_budget_sec: float = 10.0 * 3600
    reserve_sec: float = 20 * 60

    # ---- io ----
    output_dir: str = "/kaggle/working/qwen_lora_adapter"

    label2name = {0: "winner_model_a", 1: "winner_model_b", 2: "winner_tie"}
    name2label = {v: k for k, v in label2name.items()}
    class_labels = [0, 1, 2]
    class_names = ["winner_model_a", "winner_model_b", "winner_tie"]


HARD_DEADLINE = NOTEBOOK_T0 + CFG.time_budget_sec
TRAIN_DEADLINE = HARD_DEADLINE - CFG.reserve_sec


def elapsed():
    return time.time() - NOTEBOOK_T0


def fmt_time(s):
    s = int(max(0, s))
    h, r = divmod(s, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h {m:02d}m {s:02d}s"


def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True


seed_everything(CFG.seed)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA device. Set Accelerator -> GPU T4 x2 in notebook settings.")

DEVICE = torch.device("cuda")
print("GPU          :", torch.cuda.get_device_name(0))
print("GPU memory   :", f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("visible GPUs :", torch.cuda.device_count())
print("time budget  :", fmt_time(CFG.time_budget_sec))


def make_autocast():
    try:
        return torch.amp.autocast(device_type="cuda", dtype=torch.float16)
    except (TypeError, AttributeError):
        return torch.cuda.amp.autocast()


def make_scaler():
    try:
        return torch.amp.GradScaler("cuda")
    except (TypeError, AttributeError):
        return torch.cuda.amp.GradScaler()

## Environment self-test

Exercises the exact stack — 4-bit quantization, LoRA, AMP, `unscale_` — on a tiny random model.
~30 seconds. If any package combination is broken, we find out now rather than at hour 9.

In [ ]:
def environment_selftest():
    from transformers import Qwen2Config, Qwen2ForSequenceClassification

    cfg = Qwen2Config(
        vocab_size=256, hidden_size=64, num_hidden_layers=2, num_attention_heads=4,
        num_key_value_heads=2, intermediate_size=128, max_position_embeddings=128,
        num_labels=3, pad_token_id=0,
    )
    m = Qwen2ForSequenceClassification(cfg).float().cuda()

    names = {n.split(".")[-1] for n, mod in m.named_modules() if isinstance(mod, nn.Linear)}
    need = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"}
    assert need <= names, f"unexpected Qwen layer names: {sorted(names)}"

    lc = LoraConfig(r=4, lora_alpha=8, lora_dropout=0.0, bias="none",
                    task_type=TaskType.SEQ_CLS, target_modules=sorted(need),
                    modules_to_save=["score"], inference_mode=False)
    m = get_peft_model(m, lc)

    trainable = {n for n, p in m.named_parameters() if p.requires_grad}
    assert any("lora_" in n for n in trainable), "no LoRA params trainable"
    assert any("score" in n for n in trainable), "score head is frozen -- modules_to_save failed"

    opt = torch.optim.AdamW([p for p in m.parameters() if p.requires_grad], lr=1e-4)
    scaler = make_scaler()
    ids = torch.randint(1, 256, (2, 16), device="cuda")
    msk = torch.ones(2, 16, dtype=torch.long, device="cuda")

    with make_autocast():
        out = m(input_ids=ids, attention_mask=msk).logits
        loss = nn.CrossEntropyLoss()(out.float(), torch.tensor([0, 1], device="cuda"))
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_([p for p in m.parameters() if p.requires_grad], 1.0)
    scaler.step(opt)
    scaler.update()

    assert tuple(out.shape) == (2, 3), out.shape
    print("Self-test PASSED: transformers + peft(SEQ_CLS) + modules_to_save + AMP + unscale_ all OK")

    del m, opt, scaler
    gc.collect()
    torch.cuda.empty_cache()


environment_selftest()

---
# 3 · Data

In [ ]:
def find_data_dir():
    root = "/kaggle/input"
    if os.path.isdir(root):
        print("Contents of /kaggle/input:", sorted(os.listdir(root)))
    candidates = ["/kaggle/input/llm-classification-finetuning",
                  "/kaggle/input/competitions/llm-classification-finetuning",
                  "./data", "."]
    if os.path.isdir(root):
        for d in sorted(os.listdir(root)):
            sub = os.path.join(root, d)
            candidates.append(sub)
            if os.path.isdir(sub):
                candidates += [os.path.join(sub, s) for s in sorted(os.listdir(sub))
                               if os.path.isdir(os.path.join(sub, s))]
    for c in candidates:
        if os.path.exists(os.path.join(c, "train.csv")):
            return c
    raise FileNotFoundError("train.csv not found. Searched:\n  " + "\n  ".join(candidates))


DATA_DIR = find_data_dir()
print("Using data directory:", DATA_DIR)

df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
print("train.csv:", df.shape)

## Preprocessing

Unlike the DeBERTa notebook, this uses **every conversation turn**, not just the first, and puts the
prompt and both responses in a **single** sequence so the model can compare them directly with
attention. That single-sequence framing is what the strong solutions to this competition used.

Long fields are truncated **head + tail** — the opening of an answer establishes its approach and the
ending reveals whether it actually finished, while the middle is the most expendable part.

In [ ]:
def parse_list(x):
    """String-formatted list -> list of strings. Handles `null` and never raises."""
    if isinstance(x, list):
        items = x
    elif not isinstance(x, str):
        return [""]
    else:
        items = None
        for loader in (json.loads, __import__("ast").literal_eval):
            try:
                items = loader(x)
                break
            except Exception:
                continue
        if items is None:
            return [""]
    if not isinstance(items, (list, tuple)):
        items = [items]
    return ["" if v is None else str(v) for v in items] or [""]


def head_tail(text, budget, head_frac=0.55):
    """Keep the start and the end of an over-long field, drop the middle."""
    if len(text) <= budget:
        return text
    head = int(budget * head_frac)
    tail = budget - head
    return text[:head] + "\n...[truncated]...\n" + text[-tail:]


def build_text(row):
    prompts = parse_list(row["prompt"])
    resp_a = parse_list(row["response_a"])
    resp_b = parse_list(row["response_b"])
    n = max(len(prompts), len(resp_a), len(resp_b))

    def get(lst, i):
        return lst[i] if i < len(lst) else ""

    # Per-turn budget: with several turns each one gets a smaller share.
    turns = max(1, min(n, 3))
    pb = max(150, CFG.prompt_chars // turns)
    rb = max(250, CFG.response_chars // turns)

    parts = []
    for i in range(min(n, turns)):
        p = head_tail(get(prompts, i), pb)
        a = head_tail(get(resp_a, i), rb)
        b = head_tail(get(resp_b, i), rb)
        prefix = f"## Round {i + 1}\n" if turns > 1 else ""
        parts.append(f"{prefix}Prompt: {p}\n\nResponse A: {a}\n\nResponse B: {b}")

    body = "\n\n".join(parts)
    return (
        "You are judging which chatbot response a human would prefer.\n\n"
        f"{body}\n\n"
        "Which response is better: A, B, or a tie?"
    )


t0 = time.time()
df["text"] = df.apply(build_text, axis=1)
df["label"] = df[["winner_model_a", "winner_model_b", "winner_tie"]].values.argmax(axis=1)
print(f"built inputs in {time.time() - t0:.1f}s")

print("\nchar length: mean %.0f  p95 %.0f  max %.0f"
      % (df.text.str.len().mean(), df.text.str.len().quantile(0.95), df.text.str.len().max()))
print("\nlabel distribution:")
print(df.label.value_counts().sort_index())
print("\n" + "=" * 70)
print(df.text.iloc[0][:900])
print("=" * 70)

---
# 4 · Tokenizer, split, dataset

Padding is **right**-side: HuggingFace's decoder classifiers locate the pooling position by finding
the first pad token, which assumes right padding. Getting this backwards silently pools garbage.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("tokenizer :", type(tokenizer).__name__)
print("vocab     :", len(tokenizer))
print("pad token :", tokenizer.pad_token, tokenizer.pad_token_id)

train_df, valid_df = train_test_split(
    df, test_size=CFG.validation_size, stratify=df["label"], random_state=CFG.seed
)
train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True).head(CFG.max_val_samples)
print(f"\ntrain {len(train_df):,} | valid {len(valid_df):,} (capped at {CFG.max_val_samples})")


def tokenize_frame(frame, desc):
    ids = []
    texts = frame.text.tolist()
    for i in tqdm(range(0, len(texts), 512), desc=desc):
        enc = tokenizer(texts[i:i + 512], truncation=True, max_length=CFG.max_length,
                        padding=False, add_special_tokens=True)
        ids.extend(enc["input_ids"])
    return ids


t0 = time.time()
train_ids = tokenize_frame(train_df, "tokenize train")
valid_ids = tokenize_frame(valid_df, "tokenize valid")
print(f"tokenised in {fmt_time(time.time() - t0)}")

_lens = np.array([len(x) for x in train_ids])
print(f"token length: mean {_lens.mean():.0f} | p95 {np.percentile(_lens, 95):.0f} "
      f"| at cap {(_lens >= CFG.max_length).mean():.1%}")


class PairDataset(Dataset):
    def __init__(self, input_ids, labels=None):
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        item = {"input_ids": self.input_ids[i]}
        if self.labels is not None:
            item["label"] = int(self.labels[i])
        return item


class PadCollator:
    """Dynamic padding: pad each batch to its own longest sequence, not to max_length."""

    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, batch):
        maxlen = max(len(b["input_ids"]) for b in batch)
        ids = torch.full((len(batch), maxlen), self.pad_id, dtype=torch.long)
        msk = torch.zeros((len(batch), maxlen), dtype=torch.long)
        for i, b in enumerate(batch):
            n = len(b["input_ids"])
            ids[i, :n] = torch.tensor(b["input_ids"], dtype=torch.long)
            msk[i, :n] = 1
        out = {"input_ids": ids, "attention_mask": msk}
        if "label" in batch[0]:
            out["labels"] = torch.tensor([b["label"] for b in batch], dtype=torch.long)
        return out


class LengthGroupedBatchSampler(Sampler):
    """Batch similar-length sequences together, then shuffle batch order.

    With dynamic padding this is a large real speedup: batching a 200-token sample with a
    768-token one forces the short one to be padded to 768 and wastes that compute.
    """

    def __init__(self, lengths, batch_size, shuffle=True, seed=42):
        self.lengths = np.asarray(lengths)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        idx = np.argsort(self.lengths, kind="stable")
        if self.shuffle:
            # jitter within a window so the order is not identical every epoch
            noise = rng.integers(0, 64, size=len(idx))
            idx = idx[np.argsort(self.lengths[idx] + noise, kind="stable")]
        batches = [idx[i:i + self.batch_size].tolist()
                   for i in range(0, len(idx), self.batch_size)]
        if self.shuffle:
            rng.shuffle(batches)
        self.epoch += 1
        return iter(batches)

    def __len__(self):
        return math.ceil(len(self.lengths) / self.batch_size)


collator = PadCollator(tokenizer.pad_token_id)
train_dataset = PairDataset(train_ids, train_df.label.values)
valid_dataset = PairDataset(valid_ids, valid_df.label.values)

train_loader = DataLoader(
    train_dataset,
    batch_sampler=LengthGroupedBatchSampler(_lens, CFG.batch_size, shuffle=True, seed=CFG.seed),
    collate_fn=collator, num_workers=2, pin_memory=True,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_sampler=LengthGroupedBatchSampler([len(x) for x in valid_ids],
                                            CFG.eval_batch_size, shuffle=False),
    collate_fn=collator, num_workers=2, pin_memory=True,
)

_b = next(iter(train_loader))
print("\nbatch:", {k: tuple(v.shape) for k, v in _b.items()})
del _b

---
# 5 · Model — 4-bit Qwen2.5-7B + LoRA

**fp16, not bf16.** The T4 is compute capability 7.5; bf16 needs 8.0+. Every QLoRA recipe you find
online defaults to bf16 and will silently fall back or error here.

The `score` head does not exist in the checkpoint, so it is randomly initialised and **must** train —
`modules_to_save=["score"]` tells PEFT to keep it trainable and to save it alongside the adapter.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model_kwargs = dict(
    num_labels=CFG.num_labels,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)
# transformers v5 renamed `torch_dtype` -> `dtype`; support both.
try:
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name, dtype=torch.float16, **model_kwargs)
except TypeError:
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name, torch_dtype=torch.float16, **model_kwargs)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
_present = {n.split(".")[-1] for n, m in model.named_modules() if isinstance(m, nn.Linear)}
TARGET_MODULES = [t for t in TARGET_MODULES if t in _present]
print("LoRA targets:", TARGET_MODULES)

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=TARGET_MODULES,
    modules_to_save=["score"],
    inference_mode=False,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

_total = sum(p.numel() for p in model.parameters())
_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
_lora = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and "lora_" in n)
_head = sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and "score" in n)
print(f"  Total parameters     : {_total:,}")
print(f"  Trainable parameters : {_train:,}  (LoRA {_lora:,} + score head {_head:,})")
print(f"  Trainable percentage : {100 * _train / _total:.4f}%")
assert _head > 0, "score head is frozen -- it is randomly initialised and must train"
print(f"  GPU memory after load: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

---
# 6 · Optimizer, scheduler, loops

In [ ]:
steps_per_epoch = math.ceil(len(train_loader) / CFG.gradient_accumulation_steps)
total_steps = steps_per_epoch * CFG.epochs

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG.learning_rate, weight_decay=CFG.weight_decay,
)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(CFG.warmup_ratio * total_steps)),
    num_training_steps=total_steps,
)
scaler = make_scaler()
criterion = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)

print(f"micro-batches/epoch {len(train_loader):,} | optimiser steps {total_steps:,}")
print(f"effective batch     {CFG.batch_size * CFG.gradient_accumulation_steps}")
print(f"train deadline      {fmt_time(CFG.time_budget_sec - CFG.reserve_sec)} from start")


@torch.no_grad()
def validate():
    model.eval()
    probs, ys = [], []
    for batch in tqdm(valid_loader, desc="valid", leave=False):
        ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        msk = batch["attention_mask"].to(DEVICE, non_blocking=True)
        with make_autocast():
            logits = model(input_ids=ids, attention_mask=msk).logits
        probs.append(torch.softmax(logits.float(), -1).cpu().numpy())
        ys.append(batch["labels"].numpy())
    model.train()
    p = np.concatenate(probs).astype(np.float64)
    p = p / p.sum(1, keepdims=True)
    y = np.concatenate(ys)
    return log_loss(y, p, labels=CFG.class_labels), accuracy_score(y, p.argmax(1))


def save_adapter(tag=""):
    os.makedirs(CFG.output_dir, exist_ok=True)
    model.save_pretrained(CFG.output_dir)
    tokenizer.save_pretrained(CFG.output_dir)
    with open(os.path.join(CFG.output_dir, "run_info.json"), "w") as f:
        json.dump({"base_model": CFG.model_name, "max_length": CFG.max_length,
                   "prompt_chars": CFG.prompt_chars, "response_chars": CFG.response_chars,
                   "best_val_log_loss": BEST["log_loss"], "step": BEST["step"],
                   "samples_seen": BEST["samples"], "tag": tag}, f, indent=2)
    mb = sum(os.path.getsize(os.path.join(CFG.output_dir, f))
             for f in os.listdir(CFG.output_dir)
             if os.path.isfile(os.path.join(CFG.output_dir, f))) / 1024**2
    return mb

---
# 7 · Training

Validates every `eval_every_steps` optimiser steps and saves the adapter whenever validation log loss
improves. The hard deadline ends training cleanly, so even a stop at hour 10 leaves a usable model on
disk — this run is expected to finish on the clock, not on the epoch.

In [ ]:
BEST = {"log_loss": float("inf"), "acc": 0.0, "step": -1, "samples": 0}
HISTORY = []

model.train()
optimizer.zero_grad(set_to_none=True)

running, seen, opt_step = 0.0, 0, 0
stop = False
t_train = time.time()

print(f"baseline (uniform prior) log loss = {math.log(3):.5f}\n")

pbar = tqdm(train_loader, desc="train", total=len(train_loader))
for micro, batch in enumerate(pbar):
    ids = batch["input_ids"].to(DEVICE, non_blocking=True)
    msk = batch["attention_mask"].to(DEVICE, non_blocking=True)
    y = batch["labels"].to(DEVICE, non_blocking=True)

    with make_autocast():
        logits = model(input_ids=ids, attention_mask=msk).logits
        loss = criterion(logits.float(), y)

    scaler.scale(loss / CFG.gradient_accumulation_steps).backward()
    running += loss.item() * y.size(0)
    seen += y.size(0)

    if (micro + 1) % CFG.gradient_accumulation_steps == 0 or (micro + 1) == len(train_loader):
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], CFG.max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        opt_step += 1

        if opt_step % 10 == 0:
            pbar.set_postfix(loss=f"{running / max(1, seen):.4f}",
                             lr=f"{optimizer.param_groups[0]['lr']:.2e}",
                             seen=f"{seen:,}", el=fmt_time(elapsed()))

        if opt_step % CFG.eval_every_steps == 0 or time.time() > TRAIN_DEADLINE:
            ll, acc = validate()
            flag = ""
            if ll < BEST["log_loss"]:
                BEST.update({"log_loss": ll, "acc": acc, "step": opt_step, "samples": seen})
                mb = save_adapter("best")
                flag = f"  <-- best, saved {mb:.1f} MB"
            HISTORY.append({"step": opt_step, "samples": seen,
                            "train_loss": running / max(1, seen),
                            "val_log_loss": ll, "val_acc": acc, "elapsed": elapsed()})
            print(f"\nstep {opt_step:,} | samples {seen:,} | train {running / max(1, seen):.4f} "
                  f"| val log loss {ll:.5f} | val acc {acc:.4f} | {fmt_time(elapsed())}{flag}")

            if time.time() > TRAIN_DEADLINE:
                print("\n!! time budget reached -- stopping training cleanly.")
                stop = True

    if stop:
        break

TRAIN_SEC = time.time() - t_train
print(f"\ntraining finished in {fmt_time(TRAIN_SEC)}")

---
# 8 · Final validation and save

In [ ]:
if BEST["step"] < 0:
    print("No validation ran; saving current weights as a fallback.")
    ll, acc = validate()
    BEST.update({"log_loss": ll, "acc": acc, "step": opt_step, "samples": seen})
    save_adapter("final")

size_mb = save_adapter("final")

print("=" * 72)
print("RESULT")
print("=" * 72)
print(f"  Base model            : {CFG.model_name}")
print(f"  Best val log loss     : {BEST['log_loss']:.5f}   (uniform prior {math.log(3):.5f})")
print(f"  Best val accuracy     : {BEST['acc']:.4f}")
print(f"  At optimiser step     : {BEST['step']:,}")
print(f"  Training samples seen : {BEST['samples']:,} of {len(train_dataset):,} "
      f"({100 * BEST['samples'] / len(train_dataset):.1f}% of one epoch)")
print(f"  Adapter directory     : {CFG.output_dir}  ({size_mb:.1f} MB)")
print(f"  Training time         : {fmt_time(TRAIN_SEC)}")
print(f"  Total notebook time   : {fmt_time(elapsed())}")
print(f"  Peak GPU memory       : {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("=" * 72)

hist = pd.DataFrame(HISTORY)
if len(hist):
    display(hist)
    hist.to_csv("/kaggle/working/training_history.csv", index=False)

print("\nFiles in", CFG.output_dir)
for f in sorted(os.listdir(CFG.output_dir)):
    print(f"  {f:40s} {os.path.getsize(os.path.join(CFG.output_dir, f)) / 1024**2:8.2f} MB")

---
# Next step — run the inference notebook

1. **Save Version** on this notebook and let it finish, so `/kaggle/working/qwen_lora_adapter/`
   becomes a committed output.
2. Open the companion notebook **`lmsys-qwen7b-qlora-infer.ipynb`**.
3. Add this notebook's output as an input: *Add Input → Notebook Output → this notebook*.
4. Run it and **Submit to Competition** from there. It loads the base model plus this adapter and
   predicts the hidden test set in ~3–4h, inside Kaggle's 9h submission limit.

The adapter is only a few hundred MB — the 15 GB base model is downloaded fresh in the inference
notebook rather than copied, which is the whole storage argument for LoRA.